# 01. 탐색적 데이터 분석 (EDA)

**목적**: 국민건강보험공단 건강검진 데이터를 분석하여  
- 결측치 및 이상치 파악  
- 주요 건강 수치의 분포 확인  
- 정상/주의/위험 기준 적용 가능 여부 검증  
- 02_clustering 및 03_classification 설계를 위한 근거 마련

## 0. 라이브러리 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'  # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
# CSV 파일 경로를 실제 경로로 수정하세요
DATA_PATH = '../data/국민건강보험공단_건강검진정보_2024.CSV'

df = pd.read_csv(DATA_PATH, encoding='cp949')
print(f'데이터 shape: {df.shape}')
df.head()

ModuleNotFoundError: No module named 'seaborn'

## 1. 컬럼명 정리 및 표준화

In [ ]:
# 원본 컬럼 확인
print('원본 컬럼 목록:')
for col in df.columns:
    print(f'  {col}')

In [ ]:
# 컬럼명 표준화 (실제 CSV 컬럼명에 맞게 수정 필요)
COLUMN_MAP = {
    '성별코드': 'gender',
    '연령대코드(5세단위)': 'age_group',
    '신장(5Cm단위)': 'height',
    '체중(5Kg단위)': 'weight',
    '허리둘레': 'waist',
    '수축기혈압': 'sbp',        # Systolic Blood Pressure
    '이완기혈압': 'dbp',        # Diastolic Blood Pressure
    '식전혈당(공복혈당)': 'fbs', # Fasting Blood Sugar
    '총콜레스테롤': 'total_chol',
    '트리글리세라이드': 'tg',
    'HDL콜레스테롤': 'hdl',
    'LDL콜레스테롤': 'ldl',
    '흡연상태': 'smoke',
    '음주여부': 'alcohol',
}

# 실제 존재하는 컬럼만 매핑
existing_map = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
df = df.rename(columns=existing_map)

# 분석에 사용할 핵심 컬럼
CORE_COLS = ['gender', 'age_group', 'height', 'weight', 'waist',
             'sbp', 'dbp', 'fbs', 'total_chol', 'tg', 'hdl', 'ldl']

# 실제 존재하는 핵심 컬럼만 추출
available_core = [c for c in CORE_COLS if c in df.columns]
print(f'사용 가능한 핵심 컬럼: {available_core}')

## 2. 기본 데이터 정보

In [ ]:
print('=== 기본 정보 ===')
print(f'총 레코드 수: {len(df):,}명')
print(f'총 컬럼 수: {df.shape[1]}개')
print()
print('=== 데이터 타입 ===')
print(df[available_core].dtypes)

In [ ]:
# 기술 통계
df[available_core].describe().round(2)

## 3. 결측치 분석

In [ ]:
# 결측치 수 및 비율
missing = pd.DataFrame({
    '결측치 수': df[available_core].isnull().sum(),
    '결측 비율(%)': (df[available_core].isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['결측치 수'] > 0].sort_values('결측 비율(%)', ascending=False)

print('=== 결측치 현황 ===')
print(missing if len(missing) > 0 else '결측치 없음')

In [ ]:
# 결측치 시각화
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    missing['결측 비율(%)'].plot(kind='barh', ax=ax, color='#E74C3C')
    ax.set_title('컬럼별 결측 비율(%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('결측 비율(%)')
    for i, v in enumerate(missing['결측 비율(%)']):
        ax.text(v + 0.2, i, f'{v:.1f}%', va='center')
    plt.tight_layout()
    plt.show()
    
    print('\n[판단]')
    for col, row in missing.iterrows():
        rate = row['결측 비율(%)']
        if rate > 30:
            print(f'  {col}: 결측 {rate:.1f}% → 클러스터링 제외 권장')
        elif rate > 10:
            print(f'  {col}: 결측 {rate:.1f}% → 중앙값 대체 검토 필요')
        else:
            print(f'  {col}: 결측 {rate:.1f}% → 중앙값 대체 가능')

## 4. 이상치 분석

In [ ]:
# 의학적으로 불가능한 수치 기준
OUTLIER_BOUNDS = {
    'sbp':        (60, 300),
    'dbp':        (40, 200),
    'fbs':        (50, 600),
    'total_chol': (50, 600),
    'tg':         (20, 3000),
    'hdl':        (10, 200),
    'ldl':        (10, 500),
    'height':     (100, 220),
    'weight':     (20, 300),
    'waist':      (40, 200),
}

print('=== 의학적 이상치 현황 ===')
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    if col not in df.columns:
        continue
    out = df[(df[col] < lo) | (df[col] > hi)][col]
    print(f'{col:12s}: {len(out):6,}건 ({len(out)/len(df)*100:.2f}%)  범위 기준: {lo}~{hi}')

In [ ]:
# 박스플롯으로 이상치 시각화
numeric_cols = [c for c in ['sbp', 'dbp', 'fbs', 'total_chol', 'hdl', 'ldl', 'tg'] if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#3498DB', alpha=0.6))
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xticks([])

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('주요 건강 수치 박스플롯 (이상치 확인)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 주요 건강 수치 분포

In [ ]:
# 분포 히스토그램
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

plot_configs = [
    ('sbp',        '수축기 혈압 (mmHg)',  [120, 140],  ['#2ECC71', '#F39C12', '#E74C3C']),
    ('dbp',        '이완기 혈압 (mmHg)',  [80, 90],    ['#2ECC71', '#F39C12', '#E74C3C']),
    ('fbs',        '공복혈당 (mg/dL)',    [100, 126],  ['#2ECC71', '#F39C12', '#E74C3C']),
    ('total_chol', '총콜레스테롤 (mg/dL)',[200, 240],  ['#2ECC71', '#F39C12', '#E74C3C']),
    ('hdl',        'HDL (mg/dL)',         [],          ['#3498DB']),
    ('ldl',        'LDL (mg/dL)',         [],          ['#3498DB']),
    ('tg',         '중성지방 (mg/dL)',    [],          ['#3498DB']),
]

for i, (col, label, vlines, colors) in enumerate(plot_configs):
    if col not in df.columns:
        axes[i].set_visible(False)
        continue
    data = df[col].dropna()
    axes[i].hist(data, bins=50, color=colors[0], alpha=0.7, edgecolor='white')
    for vl in vlines:
        axes[i].axvline(vl, color='red', linestyle='--', linewidth=1.5, label=str(vl))
    axes[i].set_title(label, fontsize=11)
    axes[i].set_ylabel('빈도')
    if vlines:
        axes[i].legend(fontsize=8)

for j in range(len(plot_configs), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('주요 건강 수치 분포 (빨간 점선: 정상/주의/위험 기준)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 정상/주의/위험 분류 적용

In [ ]:
# BMI 계산 (신장 단위: cm)
if 'height' in df.columns and 'weight' in df.columns:
    df['bmi'] = (df['weight'] / ((df['height'] / 100) ** 2)).round(1)

# 분류 함수 정의
def classify_sbp(v):
    if pd.isna(v): return np.nan
    if v < 120: return '정상'
    elif v < 140: return '주의'
    else: return '위험'

def classify_dbp(v):
    if pd.isna(v): return np.nan
    if v < 80: return '정상'
    elif v < 90: return '주의'
    else: return '위험'

def classify_fbs(v):
    if pd.isna(v): return np.nan
    if v < 100: return '정상'
    elif v < 126: return '주의'
    else: return '위험'

def classify_chol(v):
    if pd.isna(v): return np.nan
    if v < 200: return '정상'
    elif v < 240: return '주의'
    else: return '위험'

def classify_bmi(v):
    if pd.isna(v): return np.nan
    if 18.5 <= v < 23: return '정상'
    elif v < 25: return '주의'
    else: return '위험'

def classify_waist(row):
    if pd.isna(row.get('waist')): return np.nan
    gender = row.get('gender')
    waist = row['waist']
    if gender == 1:   # 남성
        return '위험' if waist >= 90 else '정상'
    elif gender == 2: # 여성
        return '위험' if waist >= 85 else '정상'
    return np.nan

# 분류 컬럼 생성
if 'sbp' in df.columns:        df['sbp_class']   = df['sbp'].apply(classify_sbp)
if 'dbp' in df.columns:        df['dbp_class']   = df['dbp'].apply(classify_dbp)
if 'fbs' in df.columns:        df['fbs_class']   = df['fbs'].apply(classify_fbs)
if 'total_chol' in df.columns: df['chol_class']  = df['total_chol'].apply(classify_chol)
if 'bmi' in df.columns:        df['bmi_class']   = df['bmi'].apply(classify_bmi)
if 'waist' in df.columns and 'gender' in df.columns:
    df['waist_class'] = df.apply(classify_waist, axis=1)

print('분류 컬럼 생성 완료')

In [ ]:
# 분류 결과 비율 요약
class_cols = {
    'sbp_class':   '수축기 혈압',
    'dbp_class':   '이완기 혈압',
    'fbs_class':   '공복혈당',
    'chol_class':  '총콜레스테롤',
    'bmi_class':   'BMI',
    'waist_class': '허리둘레',
}

print('=== 정상/주의/위험 분류 비율 ===')
for col, label in class_cols.items():
    if col not in df.columns:
        continue
    counts = df[col].value_counts()
    total = counts.sum()
    print(f'\n[{label}]')
    for cls in ['정상', '주의', '위험']:
        n = counts.get(cls, 0)
        print(f'  {cls}: {n:7,}명 ({n/total*100:.1f}%)')

In [ ]:
# 분류 비율 시각화
available_class = [(col, label) for col, label in class_cols.items() if col in df.columns]
n = len(available_class)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

colors = {'정상': '#2ECC71', '주의': '#F39C12', '위험': '#E74C3C'}

for i, (col, label) in enumerate(available_class):
    counts = df[col].value_counts().reindex(['정상', '주의', '위험']).fillna(0)
    pcts = counts / counts.sum() * 100
    bar_colors = [colors.get(c, '#95A5A6') for c in counts.index]
    bars = axes[i].bar(counts.index, pcts, color=bar_colors, edgecolor='white', linewidth=0.5)
    for bar, pct in zip(bars, pcts):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
    axes[i].set_title(label, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('비율(%)')
    axes[i].set_ylim(0, 100)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('항목별 정상/주의/위험 분류 비율', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 성별·연령대별 분포

In [ ]:
# 성별 분포
if 'gender' in df.columns:
    gender_map = {1: '남성', 2: '여성'}
    df['gender_label'] = df['gender'].map(gender_map)
    print('=== 성별 분포 ===')
    print(df['gender_label'].value_counts())

In [ ]:
# 연령대별 주요 수치 평균
if 'age_group' in df.columns:
    # 연령대 코드를 실제 나이로 변환 (5세 단위: 코드 * 5 = 나이 하한)
    numeric_health = [c for c in ['sbp', 'dbp', 'fbs', 'total_chol', 'bmi'] if c in df.columns]
    age_group_mean = df.groupby('age_group')[numeric_health].mean().round(1)
    print('=== 연령대별 평균 건강 수치 ===')
    print(age_group_mean)

In [ ]:
# 연령대별 혈당 분포 시각화
if 'age_group' in df.columns and 'fbs' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 연령대별 혈당 박스플롯
    age_groups = sorted(df['age_group'].dropna().unique())
    data_by_age = [df[df['age_group'] == ag]['fbs'].dropna().values for ag in age_groups]
    axes[0].boxplot(data_by_age, labels=age_groups, patch_artist=True,
                    boxprops=dict(facecolor='#3498DB', alpha=0.6))
    axes[0].axhline(100, color='orange', linestyle='--', label='주의 기준(100)')
    axes[0].axhline(126, color='red', linestyle='--', label='위험 기준(126)')
    axes[0].set_title('연령대별 공복혈당 분포', fontsize=12)
    axes[0].set_xlabel('연령대 코드')
    axes[0].set_ylabel('공복혈당 (mg/dL)')
    axes[0].legend()

    # 연령대별 BMI 박스플롯
    if 'bmi' in df.columns:
        data_bmi = [df[df['age_group'] == ag]['bmi'].dropna().values for ag in age_groups]
        axes[1].boxplot(data_bmi, labels=age_groups, patch_artist=True,
                        boxprops=dict(facecolor='#9B59B6', alpha=0.6))
        axes[1].axhline(23, color='orange', linestyle='--', label='주의 기준(23)')
        axes[1].axhline(25, color='red', linestyle='--', label='위험 기준(25)')
        axes[1].set_title('연령대별 BMI 분포', fontsize=12)
        axes[1].set_xlabel('연령대 코드')
        axes[1].set_ylabel('BMI')
        axes[1].legend()

    plt.tight_layout()
    plt.show()

## 8. 상관관계 분석

In [ ]:
# 주요 건강 수치 간 상관계수
corr_cols = [c for c in ['sbp', 'dbp', 'fbs', 'total_chol', 'hdl', 'ldl', 'tg', 'bmi', 'waist'] if c in df.columns]
corr_matrix = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('주요 건강 수치 상관관계 히트맵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[클러스터링 변수 선택 참고]')
print('상관계수 0.7 이상인 변수 쌍 (다중공선성 주의):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = abs(corr_matrix.iloc[i, j])
        if val >= 0.7:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {val:.2f}')

## 9. 복합 위험 패턴 분석

In [ ]:
# 복합 위험 조합 빈도 (혈당 + 혈압 동시 이상)
if 'fbs_class' in df.columns and 'sbp_class' in df.columns:
    combo = df.groupby(['fbs_class', 'sbp_class']).size().unstack(fill_value=0)
    # 퍼센트로 변환
    combo_pct = (combo / len(df) * 100).round(2)
    print('=== 혈당 × 혈압 복합 분류 비율(%) ===')
    print(combo_pct)

In [ ]:
# 위험 항목 개수 분포
risk_cols = [c for c in ['sbp_class', 'dbp_class', 'fbs_class', 'chol_class', 'bmi_class', 'waist_class'] if c in df.columns]

df['risk_count'] = (df[risk_cols] == '위험').sum(axis=1)
df['caution_count'] = (df[risk_cols] == '주의').sum(axis=1)

print('=== 위험 항목 개수 분포 ===')
print(df['risk_count'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['risk_count'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='#E74C3C', edgecolor='white')
axes[0].set_title('위험 항목 동시 보유 수 분포', fontsize=12)
axes[0].set_xlabel('위험 항목 수')
axes[0].set_ylabel('인원 수')
axes[0].tick_params(axis='x', rotation=0)

df['caution_count'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#F39C12', edgecolor='white')
axes[1].set_title('주의 항목 동시 보유 수 분포', fontsize=12)
axes[1].set_xlabel('주의 항목 수')
axes[1].set_ylabel('인원 수')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 10. EDA 결과 요약 및 다음 단계 제언

In [ ]:
print('=' * 60)
print('EDA 결과 요약')
print('=' * 60)

print(f'\n[데이터 규모]')
print(f'  총 {len(df):,}명')

print(f'\n[결측치]')
for col in available_core:
    rate = df[col].isnull().sum() / len(df) * 100
    if rate > 0:
        print(f'  {col}: {rate:.1f}%')

print(f'\n[02_clustering 사용 권장 변수]')
cluster_vars = []
for col in ['sbp', 'dbp', 'fbs', 'bmi', 'waist', 'total_chol', 'hdl', 'ldl', 'tg']:
    if col in df.columns:
        missing_rate = df[col].isnull().sum() / len(df) * 100
        if missing_rate < 30:
            cluster_vars.append(col)
            print(f'  ✓ {col} (결측 {missing_rate:.1f}%)')
        else:
            print(f'  ✗ {col} (결측 {missing_rate:.1f}% → 제외 권장)')

print(f'\n[03_classification 타겟 변수]')
print('  → 02_clustering 결과 군집 레이블을 타겟으로 활용')
print('  → 또는 맞춤형 식단 데이터의 Recommended_Meal_Plan을 타겟으로 활용')

print('\n' + '=' * 60)

In [ ]:
# 전처리된 데이터 저장 (02_clustering에서 사용)
save_cols = [c for c in cluster_vars + ['gender', 'age_group'] + 
             [c for c in df.columns if c.endswith('_class')] if c in df.columns]

df_clean = df[save_cols].copy()

# 결측치 제거 (핵심 수치 기준)
key_cols = [c for c in ['sbp', 'dbp', 'fbs', 'bmi'] if c in df_clean.columns]
df_clean = df_clean.dropna(subset=key_cols)

# 의학적 이상치 제거
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    if col in df_clean.columns:
        df_clean = df_clean[(df_clean[col].isna()) | ((df_clean[col] >= lo) & (df_clean[col] <= hi))]

print(f'전처리 전: {len(df):,}명')
print(f'전처리 후: {len(df_clean):,}명')
print(f'제거 비율: {(1 - len(df_clean)/len(df))*100:.1f}%')

# 저장
df_clean.to_csv('../data/health_cleaned.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: ../data/health_cleaned.csv')
print('→ 02_clustering.ipynb에서 이 파일을 사용합니다.')